In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from estimation_forecast_functions_var import DataCleaner_EUR
import holidays
from simple_strat_funcs import Clean_Implied_Vols_EUR_with_smile
from scipy.stats import norm
from hedging_strategy_class_NEW_KURT_SKEW_vega_VAR import Compare_Trading_Strategies
import seaborn as sns

/Users/alexvillamartin/Documents/MSc Diss/Code/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Global params

In [2]:
train_size = 0.7
test_size = 1 - train_size

TICKER = "EURUSD"

# always use same amount of money in USD to start and convert where necessary
IC_USD = 1_000_000 # initial capital in USD
current_rate = 1.16
NOTIONAL_USD = 1_000_000 # this is used in strategy as our notional to ensure trade positions are in domestic currency
NOTIONAL_EUR = NOTIONAL_USD / current_rate # this is used in strategy as our notional to ensure trade positions are in foreign currency

MAX_DELTA_DIFF = 500 # this param doesnt change much as our delta positions are usually much larger
SIGNAL_LB = 0.01
SIGNAL_UB = 0.99
TRANSACTION_COST_BOOL = True
TRANSACTION_COSTS_SPOT = 0.0002 / 2 # for half a leg ie selling/buying once not to enter and exit the trade
TRANSACTION_COSTS_OPTION = 0.0005 / 2 # [change to vol spreads later] for half a leg again, both in decimals

HYPERPARAM_SORT = 'sharpe_ratio' # can be 'sharpe_ratio', 'cagr', 'max_drawdown', 'total_return'
GARCH_1_2_INDICATOR = False

k_bar_MSM = 8
b_MSM = 2.0
gamma_kbar_MSM = 0.5

M = 300 # figarch lags

CONVERT_USD_INDICATOR = False # convert all portfolio values and metrics to USD for fair comparison - use when USD is not domestic

long_threshs = np.array([1.02, 1.03, 1.04, 1.05, 1.06, 1.07,  1.08, 1.09, 1.10, 1.11, 1.12, 1.13, 1.14, 1.15, 1.16, 1.17, 1.18, 1.19, 1.20, 1.21, 1.22, 1.23, 1.24, 1.25, 1.26, 1.27])
short_threshs = np.array([0.98, 0.97, 0.96, 0.95, 0.94, 0.93, 0.92,  0.91, 0.90, 0.89, 0.88, 0.87, 0.86, 0.85, 0.84, 0.83, 0.82, 0.81, 0.80, 0.79, 0.78, 0.77, 0.76, 0.75, 0.74, 0.73])
sig_multipliers = np.array([1, 3, 5, 7, 9, 11, 13, 15, 17, 20])


# Global data

In [3]:
eurusd_5m = pd.read_parquet("EUR_USD_5M.parquet").copy()[['c']]
eurusd_5m.rename(columns={'c': 'spot'}, inplace=True)  
cleaner_eurusd = DataCleaner_EUR(eurusd_5m, start_hr=6, end_hr=18, unit_test=False) 
realised_variance_eurusd = cleaner_eurusd.clean_data()
daily_log_returns_eurusd = pd.read_parquet("DF_D_EURUSD.parquet")['Log_r']
start_date = pd.to_datetime(realised_variance_eurusd.index.min())
end_date = pd.to_datetime(realised_variance_eurusd.index.max())
spot_curr = pd.read_parquet("/Users/alexvillamartin/Documents/MSc Diss/Code/DF_D_EURUSD.parquet")
overnight_domestic_rate = pd.read_csv("/Users/alexvillamartin/Documents/MSc Diss/Code/SOFR_daily.csv").set_index("date")
overnight_domestic_rate.index = (pd.to_datetime(overnight_domestic_rate.index, format='mixed').normalize())
overnight_foreign_rate = pd.read_csv("estr_daily.csv").set_index("Date")

# Global functions

Change holidays where needed. 

In [4]:
def align_spots(df, aligned_df, start, end):

    df = df[(df.index >= start) & (df.index <= end)]

    years = range(start.year, end.year + 1)

    first = holidays.US(years=years)
    second = holidays.XECB(years=years)

    hols = pd.to_datetime(list(set(first) | set(second))).normalize()

    idx  = df.index
    mask = ~idx.isin(hols)
    new_df = df.loc[mask]

    df_aligned_dates = aligned_df.index
    common_dates = new_df.index.intersection(df_aligned_dates)
    new_df = new_df.loc[common_dates]

    return new_df

def sort_rates(r_b, r_t, overn_r, aligned_df, overn_for_r):

    r_b.index = pd.to_datetime(r_b.index)
    valid_mask = ~( 
                   r_t.index.isna())
    r_t_clean = r_t[valid_mask]
    r_t_clean.index = pd.to_datetime(r_t_clean.index, format='%Y-%m-%d')
    
    valid_mask2 = ~(
                   overn_r.index.isna())
    overn_r_clean = overn_r[valid_mask2]
    overn_r_clean.index = pd.to_datetime(overn_r_clean.index, format='%Y-%m-%d')

    valid_mask3 = ~(
                   overn_for_r.index.isna())
    overn_for_r_clean = overn_for_r[valid_mask3]
    overn_for_r_clean.index = pd.to_datetime(overn_for_r_clean.index, format='mixed').normalize()    

    r_b = r_b.reindex(aligned_df.index)
    r_t_clean = r_t_clean.reindex(aligned_df.index)
    overn_r_clean = overn_r_clean.reindex(aligned_df.index)
    overn_for_r_clean = overn_for_r_clean.reindex(aligned_df.index)

    r_b = r_b.ffill()
    r_t_clean = r_t_clean.ffill()
    overn_r_clean = overn_r_clean.ffill()
    overn_for_r_clean = overn_for_r_clean.ffill()

    # convert into decimals and continously compunded versions for BSE
    r_b = pd.to_numeric(r_b['rate_pct'], errors='coerce')
    r_t_clean = pd.to_numeric(r_t_clean['rate_pct'], errors='coerce')
    overn_r_clean = pd.to_numeric(overn_r_clean['value'], errors='coerce')
    r_b = r_b / 100
    r_t_clean = r_t_clean / 100
    overn_r_clean = overn_r_clean / 100 # not continously compounded
    overn_r_clean *= 1/365 # daily

    overn_for_r_clean = pd.to_numeric(overn_for_r_clean['rate_pct'], errors='coerce')
    overn_for_r_clean = overn_for_r_clean / 100 # not continously compounded
    overn_for_r_clean *= 1/365 # daily

    r_b = np.log(1 + r_b)
    r_t_clean = np.log(1 + r_t_clean)

    return r_b, r_t_clean, overn_r_clean, overn_for_r_clean

# H=30, T=1Mo

In [5]:
OPTION_MATURITY_1 = 1 / 12
H_1 = 30 

implied_vol_data_1 = pd.read_excel('/Users/alexvillamartin/Documents/MSc Diss/Code/EURUSD_1MO_ATM_D.xlsx')
vol_smile_data_1 = pd.read_csv("eurusd_vol_smile_1mo_extra.csv").set_index("CalculationDate")
Data_clean_1 = Clean_Implied_Vols_EUR_with_smile(data=implied_vol_data_1, 
                                start_date=start_date, 
                                end_date=end_date, 
                                align_df1=realised_variance_eurusd, 
                                align_df2=daily_log_returns_eurusd, 
                                smile_df=vol_smile_data_1)
implied_vol_data_1, realised_variance_1, daily_log_returns_1, vol_smile_data_1 = Data_clean_1.get_clean_data()

N_1 = len(daily_log_returns_1)
test_align_1 = daily_log_returns_1.iloc[N_1//2:-H_1]
spot_curr_test_1 = align_spots(spot_curr, test_align_1, start_date, end_date)

r_b_1 = pd.read_csv("estr_1mo_compounded.csv").set_index("Date")
r_t_1 = pd.read_csv("/Users/alexvillamartin/Documents/MSc Diss/Code/SOFR_1mo_compounded.csv").set_index("date")[['rate_pct']]

r_b_test_1, r_t_test_1, overn_dom_r_test_1, overn_for_r_test_1= sort_rates(r_b_1, r_t_1, overnight_domestic_rate, test_align_1, overnight_foreign_rate) 
r_b_test_1.ffill(inplace=True)

In [6]:
strategy_1 = Compare_Trading_Strategies(
    return_series=daily_log_returns_1, 
    realised_variance_series=realised_variance_1,
    atm_implied_vol_data=implied_vol_data_1,
    vol_smile_data=vol_smile_data_1,
    train_size=train_size,
    ticker=TICKER,
    initial_capital_domestic=IC_USD,
    notional_base=NOTIONAL_EUR,
    maximum_delta_difference=MAX_DELTA_DIFF,
    signal_lb=SIGNAL_LB,
    signal_ub=SIGNAL_UB,
    transaction_cost_indicator=TRANSACTION_COST_BOOL,
    transaction_costs_spot=TRANSACTION_COSTS_SPOT,
    transaction_costs_option=TRANSACTION_COSTS_OPTION,
    option_maturity=OPTION_MATURITY_1,
    forecast_horizon=H_1,
    spot_series=spot_curr_test_1,
    overnight_domestic_rate=overn_dom_r_test_1,
    overnight_foreign_rate=overn_for_r_test_1,
    domestic_rate=r_t_test_1,
    foreign_rate=r_b_test_1, 
    plots = False, 
    verbose=False, 
    sort_hyperparams_by=HYPERPARAM_SORT, 
    garch_1_2_indicator=GARCH_1_2_INDICATOR, 
    kbar=k_bar_MSM,
    b=b_MSM,
    gamma_kbar=gamma_kbar_MSM,
    convert_USD=CONVERT_USD_INDICATOR, 
    M = M, 
    long_thresholds=long_threshs,
    short_thresholds=short_threshs,
    sig_multipliers=sig_multipliers)

strategy_1.prepare_universal_series()
strategy_1.get_BMSM_data()
strategy_1.get_GARCH_data()
strategy_1.get_FIGARCH_data()

Estimated parameters: m0=1.210864e+00, sigma_bar=4.854110e-01
Final log-likelihood: -8.041643e+02
Estimated parameters: m0=1.195292e+00, sigma_bar=4.820609e-01
Final log-likelihood: -1.353890e+03
Estimated parameters: omega=0.0009705549253154421, alpha=0.0378, beta=0.9598
Estimated parameters: omega=0.004347417273070562, alpha=0.0455, beta=0.9443
Estimated parameters: omega=0.04783632598877084, d=0.2740, beta=0.2590
Final log-likelihood = 598.9679
Estimated parameters: omega=0.04385120158229892, d=0.2734, beta=0.2606
Final log-likelihood = 1046.2372


In [7]:
error_metrics_df_1, log_ls, m_z_results, se_results = strategy_1.in_sample_predictions()

In [15]:
error_metrics_df_1

,Norm MSE,DM Test Stat MSE,DM p-value (one-sided) MSE,Norm MAE,DM Test Stat MAE,DM p-value (one-sided) MAE
BMSM,0.598493,NaN,NaN,0.706790,NaN,NaN
BMSM OLS,0.543508,-0.601030,0.273967,0.749637,0.742447,0.771019
GARCH,0.435715,NaN,NaN,0.576162,NaN,NaN
GARCH OLS,0.469121,0.663144,0.746317,0.666524,2.193506,0.985769
FIGARCH,0.591219,NaN,NaN,0.745043,NaN,NaN
FIGARCH OLS,0.596860,0.072235,0.528787,0.804247,1.074080,0.858499


In [ ]:
error_metrics_df_1.to_csv("USDEUR_Errors_H_30_T_1MO_OLS.csv")

In [9]:
m_z_results

{'BMSM': {'alpha_hat': -0.00436563442073568,
  'beta_hat': 1.8711495482018732,
  'alpha_p': 2.650338569546426e-09,
  'beta_p': 4.951695283165679e-10},
 'GARCH': {'alpha_hat': -0.00018490978258741417,
  'beta_hat': 1.0970272435497344,
  'alpha_p': 0.7175013473817126,
  'beta_p': 0.32364309467978514},
 'FIGARCH': {'alpha_hat': -0.003922970555464426,
  'beta_hat': 1.6597529669086173,
  'alpha_p': 2.8223549652915943e-06,
  'beta_p': 6.478130033521126e-06}}

In [10]:
log_ls

{'BMSM': np.float64(-804.1642748460371),
 'GARCH': np.float64(625.665089046425),
 'FIGARCH': np.float64(598.9679066292977)}

In [11]:
se_results

{'BMSM': array([0.02121107, 0.03239833]),
 'GARCH': array([0.00103985, 0.005942  , 0.00665643]),
 'FIGARCH': array([0.00968532, 0.04383519, 0.05231113])}

In [12]:
strategy_1.model_bmsm.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                      y   R-squared:                       0.114
Model:                            OLS   Adj. R-squared:                  0.111
Method:                 Least Squares   F-statistic:                     7.692
Date:                Sat, 30 Aug 2025   Prob (F-statistic):           4.05e-06
Time:                        19:44:16   Log-Likelihood:                -709.38
No. Observations:                1203   AIC:                             1429.
Df Residuals:                    1198   BIC:                             1454.
Df Model:                           4                                         
Covariance Type:                  HAC                                         
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.0337      0.032     -1.058      0.290      -0.096       0.029
x1             0.1566      0.075      2.087      0.037       0.010       0.304
x2            -0.2236      0.088     -2.552      0.011      -0.395      -0.052
x3            -0.0378      0.035     -1.095      0.274      -0.106       0.030
x4            -0.1640      0.044     -3.687      0.000      -0.251      -0.077
==============================================================================
Omnibus:                      329.236   Durbin-Watson:                   0.065
Prob(Omnibus):                  0.000   Jarque-Bera (JB):              891.469
Skew:                           1.411   Prob(JB):                    2.63e-194
Kurtosis:                       6.133   Cond. No.                         5.69
==============================================================================

Notes:
[1] Standard Errors are heteroscedasticity and autocorrelation robust (HAC) using 6 lags and without small sample correction
"""

In [13]:
strategy_1.model_garch.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                      y   R-squared:                       0.083
Model:                            OLS   Adj. R-squared:                  0.080
Method:                 Least Squares   F-statistic:                     19.10
Date:                Sat, 30 Aug 2025   Prob (F-statistic):           4.19e-12
Time:                        19:44:28   Log-Likelihood:                -746.85
No. Observations:                1203   AIC:                             1502.
Df Residuals:                    1199   BIC:                             1522.
Df Model:                           3                                         
Covariance Type:                  HAC                                         
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.0371      0.033     -1.110      0.267      -0.103       0.028
x1            -0.1484      0.020     -7.310      0.000      -0.188      -0.109
x2            -0.0250      0.038     -0.661      0.508      -0.099       0.049
x3            -0.0642      0.046     -1.383      0.167      -0.155       0.027
==============================================================================
Omnibus:                      440.859   Durbin-Watson:                   0.045
Prob(Omnibus):                  0.000   Jarque-Bera (JB):             1606.359
Skew:                           1.775   Prob(JB):                         0.00
Kurtosis:                       7.410   Cond. No.                         1.84
==============================================================================

Notes:
[1] Standard Errors are heteroscedasticity and autocorrelation robust (HAC) using 6 lags and without small sample correction
"""

In [14]:
strategy_1.model_figarch.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                      y   R-squared:                       0.154
Model:                            OLS   Adj. R-squared:                  0.152
Method:                 Least Squares   F-statistic:                     8.087
Date:                Sat, 30 Aug 2025   Prob (F-statistic):           2.46e-05
Time:                        19:44:37   Log-Likelihood:                -771.84
No. Observations:                1203   AIC:                             1552.
Df Residuals:                    1199   BIC:                             1572.
Df Model:                           3                                         
Covariance Type:                  HAC                                         
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.0868      0.034     -2.574      0.010      -0.153      -0.021
x1            -0.0856      0.031     -2.741      0.006      -0.147      -0.024
x2            -0.0359      0.038     -0.954      0.340      -0.110       0.038
x3            -0.2055      0.047     -4.361      0.000      -0.298      -0.113
==============================================================================
Omnibus:                      355.723   Durbin-Watson:                   0.065
Prob(Omnibus):                  0.000   Jarque-Bera (JB):             1016.742
Skew:                           1.505   Prob(JB):                    1.65e-221
Kurtosis:                       6.351   Cond. No.                         1.90
==============================================================================

Notes:
[1] Standard Errors are heteroscedasticity and autocorrelation robust (HAC) using 6 lags and without small sample correction
"""

# H=91, T=3MO

In [16]:
OPTION_MATURITY_2 = 3 / 12
H_2 = 91 

vol_smile_data_2 = pd.read_csv("eurusd_vol_smile_3mo_extra.csv").set_index("CalculationDate")
implied_vol_data_2 = pd.DataFrame({
    'Exchange Date': vol_smile_data_2.index, 
    "Bid": vol_smile_data_2['ATM'], 
    "Ask": vol_smile_data_2['ATM'],
    "BidNet": vol_smile_data_2['ATM']})
Data_clean_2 = Clean_Implied_Vols_EUR_with_smile(data=implied_vol_data_2, 
                                start_date=start_date, 
                                end_date=end_date, 
                                align_df1=realised_variance_eurusd, 
                                align_df2=daily_log_returns_eurusd, 
                                smile_df=vol_smile_data_2)
implied_vol_data_2, realised_variance_2, daily_log_returns_2, vol_smile_data_2 = Data_clean_2.get_clean_data()

N_2 = len(daily_log_returns_2)
test_align_2 = daily_log_returns_2.iloc[N_2//2:-H_2]
spot_curr_test_2 = align_spots(spot_curr, test_align_2, start_date, end_date)

r_b_2 = pd.read_csv("estr_3mo_compounded.csv").set_index("Date")
r_t_2 = pd.read_csv("/Users/alexvillamartin/Documents/MSc Diss/Code/SOFR_3mo_compounded.csv").set_index("date")[['rate_pct']]

r_b_test_2, r_t_test_2, overn_dom_r_test_2, overn_for_r_test_2= sort_rates(r_b_2, r_t_2, overnight_domestic_rate, test_align_2, overnight_foreign_rate) 
r_b_test_2.ffill(inplace=True)

In [17]:
strategy_2 = Compare_Trading_Strategies(
    return_series=daily_log_returns_2, 
    realised_variance_series=realised_variance_2,
    atm_implied_vol_data=implied_vol_data_2,
    vol_smile_data=vol_smile_data_2,
    train_size=train_size,
    ticker=TICKER,
    initial_capital_domestic=IC_USD,
    notional_base=NOTIONAL_EUR,
    maximum_delta_difference=MAX_DELTA_DIFF,
    signal_lb=SIGNAL_LB,
    signal_ub=SIGNAL_UB,
    transaction_cost_indicator=TRANSACTION_COST_BOOL,
    transaction_costs_spot=TRANSACTION_COSTS_SPOT,
    transaction_costs_option=TRANSACTION_COSTS_OPTION,
    option_maturity=OPTION_MATURITY_2,
    forecast_horizon=H_2,
    spot_series=spot_curr_test_2,
    overnight_domestic_rate=overn_dom_r_test_2,
    overnight_foreign_rate=overn_for_r_test_2,
    domestic_rate=r_t_test_2,
    foreign_rate=r_b_test_2, 
    plots = False, 
    verbose=False, 
    sort_hyperparams_by=HYPERPARAM_SORT, 
    garch_1_2_indicator=GARCH_1_2_INDICATOR, 
    kbar=k_bar_MSM,
    b=b_MSM,
    gamma_kbar=gamma_kbar_MSM,
    convert_USD=CONVERT_USD_INDICATOR, 
    M = M, 
    long_thresholds=long_threshs,
    short_thresholds=short_threshs,
    sig_multipliers=sig_multipliers)

strategy_2.prepare_universal_series()
strategy_2.get_BMSM_data()
strategy_2.get_GARCH_data()
strategy_2.get_FIGARCH_data()

Estimated parameters: m0=1.210864e+00, sigma_bar=4.854110e-01
Final log-likelihood: -8.041643e+02
Estimated parameters: m0=1.198027e+00, sigma_bar=4.821360e-01
Final log-likelihood: -1.327739e+03
Estimated parameters: omega=0.0009705549253154421, alpha=0.0378, beta=0.9598
Estimated parameters: omega=0.0011730197646743856, alpha=0.0401, beta=0.9564
Estimated parameters: omega=0.04783632598877084, d=0.2740, beta=0.2590
Final log-likelihood = 598.9679
Estimated parameters: omega=0.04362897957209833, d=0.2755, beta=0.2616
Final log-likelihood = 1017.4750


In [18]:
error_metrics_df_2, log_ls_2 , m_z_results_2, se_2 = strategy_2.in_sample_predictions()

In [19]:
error_metrics_df_2

,Norm MSE,DM Test Stat MSE,DM p-value (one-sided) MSE,Norm MAE,DM Test Stat MAE,DM p-value (one-sided) MAE
BMSM,0.720285,NaN,NaN,0.851109,NaN,NaN
BMSM OLS,0.652209,-0.694846,0.243647,0.850123,-0.017810,0.492897
GARCH,0.551689,NaN,NaN,0.747721,NaN,NaN
GARCH OLS,0.549800,-0.021767,0.491319,0.768831,0.383376,0.649244
FIGARCH,0.751109,NaN,NaN,0.925785,NaN,NaN
FIGARCH OLS,0.686041,-0.808978,0.209348,0.865296,-0.991157,0.160909


In [20]:
m_z_results_2

{'BMSM': {'alpha_hat': -0.005065168543534576,
  'beta_hat': 1.9466419766206484,
  'alpha_p': 4.921030967632268e-06,
  'beta_p': 3.673018278718203e-06},
 'GARCH': {'alpha_hat': 0.0005859288089274759,
  'beta_hat': 0.8986749667999829,
  'alpha_p': 0.3054027380536428,
  'beta_p': 0.3285730121413366},
 'FIGARCH': {'alpha_hat': -0.003308115101681588,
  'beta_hat': 1.4796131465151483,
  'alpha_p': 0.003195087650535331,
  'beta_p': 0.010389007544728846}}

In [23]:
strategy_2.model_bmsm.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                      y   R-squared:                       0.145
Model:                            OLS   Adj. R-squared:                  0.142
Method:                 Least Squares   F-statistic:                     8.577
Date:                Sat, 30 Aug 2025   Prob (F-statistic):           8.08e-07
Time:                        20:22:34   Log-Likelihood:                -451.48
No. Observations:                1142   AIC:                             913.0
Df Residuals:                    1137   BIC:                             938.2
Df Model:                           4                                         
Covariance Type:                  HAC                                         
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.0395      0.028     -1.427      0.154      -0.094       0.015
x1            -0.0532      0.064     -0.837      0.403      -0.178       0.071
x2             0.0245      0.060      0.410      0.682      -0.093       0.141
x3             0.0296      0.033      0.893      0.372      -0.035       0.095
x4            -0.1879      0.059     -3.183      0.001      -0.304      -0.072
==============================================================================
Omnibus:                       53.875   Durbin-Watson:                   0.023
Prob(Omnibus):                  0.000   Jarque-Bera (JB):               65.988
Skew:                           0.480   Prob(JB):                     4.69e-15
Kurtosis:                       3.683   Cond. No.                         7.31
==============================================================================

Notes:
[1] Standard Errors are heteroscedasticity and autocorrelation robust (HAC) using 6 lags and without small sample correction
"""

In [24]:
strategy_2.model_garch.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                      y   R-squared:                       0.053
Model:                            OLS   Adj. R-squared:                  0.051
Method:                 Least Squares   F-statistic:                     8.471
Date:                Sat, 30 Aug 2025   Prob (F-statistic):           1.44e-05
Time:                        20:22:40   Log-Likelihood:                -486.03
No. Observations:                1142   AIC:                             980.1
Df Residuals:                    1138   BIC:                             1000.
Df Model:                           3                                         
Covariance Type:                  HAC                                         
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.0841      0.028     -2.960      0.003      -0.140      -0.028
x1            -0.1112      0.024     -4.607      0.000      -0.159      -0.064
x2             0.0277      0.031      0.891      0.373      -0.033       0.089
x3            -0.0763      0.040     -1.913      0.056      -0.155       0.002
==============================================================================
Omnibus:                      165.890   Durbin-Watson:                   0.033
Prob(Omnibus):                  0.000   Jarque-Bera (JB):              273.823
Skew:                           0.947   Prob(JB):                     3.47e-60
Kurtosis:                       4.472   Cond. No.                         2.80
==============================================================================

Notes:
[1] Standard Errors are heteroscedasticity and autocorrelation robust (HAC) using 6 lags and without small sample correction
"""

In [25]:
strategy_2.model_figarch.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                      y   R-squared:                       0.211
Model:                            OLS   Adj. R-squared:                  0.209
Method:                 Least Squares   F-statistic:                     16.85
Date:                Sat, 30 Aug 2025   Prob (F-statistic):           1.03e-10
Time:                        20:22:48   Log-Likelihood:                -473.41
No. Observations:                1142   AIC:                             954.8
Df Residuals:                    1138   BIC:                             975.0
Df Model:                           3                                         
Covariance Type:                  HAC                                         
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.1139      0.028     -4.079      0.000      -0.169      -0.059
x1            -0.0497      0.035     -1.416      0.157      -0.119       0.019
x2             0.0394      0.030      1.294      0.196      -0.020       0.099
x3            -0.2320      0.036     -6.487      0.000      -0.302      -0.162
==============================================================================
Omnibus:                       45.583   Durbin-Watson:                   0.038
Prob(Omnibus):                  0.000   Jarque-Bera (JB):               53.683
Skew:                           0.443   Prob(JB):                     2.20e-12
Kurtosis:                       3.586   Cond. No.                         2.46
==============================================================================

Notes:
[1] Standard Errors are heteroscedasticity and autocorrelation robust (HAC) using 6 lags and without small sample correction
"""

# H=182, T=6MO

In [26]:
OPTION_MATURITY_3 = 6 / 12
H_3 = 182

vol_smile_data_3 = pd.read_csv("eurusd_vol_smile_6mo_extra.csv").set_index("CalculationDate")
implied_vol_data_3 = pd.DataFrame({
    'Exchange Date': vol_smile_data_3.index, 
    "Bid": vol_smile_data_3['ATM'], 
    "Ask": vol_smile_data_3['ATM'],
    "BidNet": vol_smile_data_3['ATM']})
Data_clean_3 = Clean_Implied_Vols_EUR_with_smile(data=implied_vol_data_3, 
                                start_date=start_date, 
                                end_date=end_date, 
                                align_df1=realised_variance_eurusd, 
                                align_df2=daily_log_returns_eurusd, 
                                smile_df=vol_smile_data_3)
implied_vol_data_3, realised_variance_3, daily_log_returns_3, vol_smile_data_3 = Data_clean_3.get_clean_data()

N_3 = len(daily_log_returns_3)
test_align_3 = daily_log_returns_3.iloc[N_3//2:-H_3]
spot_curr_test_3 = align_spots(spot_curr, test_align_3, start_date, end_date)

r_b_3 = pd.read_csv("estr_6mo_compounded.csv").set_index("Date")
r_t_3 = pd.read_csv("/Users/alexvillamartin/Documents/MSc Diss/Code/SOFR_6mo_compounded.csv").set_index("date")[['rate_pct']]

r_b_test_3, r_t_test_3, overn_dom_r_test_3, overn_for_r_test_3= sort_rates(r_b_3, r_t_3, overnight_domestic_rate, test_align_3, overnight_foreign_rate) 
r_b_test_3.ffill(inplace=True)

In [27]:
strategy_3 = Compare_Trading_Strategies(
    return_series=daily_log_returns_3, 
    realised_variance_series=realised_variance_3,
    atm_implied_vol_data=implied_vol_data_3,
    vol_smile_data=vol_smile_data_3,
    train_size=train_size,
    ticker=TICKER,
    initial_capital_domestic=IC_USD,
    notional_base=NOTIONAL_EUR,
    maximum_delta_difference=MAX_DELTA_DIFF,
    signal_lb=SIGNAL_LB,
    signal_ub=SIGNAL_UB,
    transaction_cost_indicator=TRANSACTION_COST_BOOL,
    transaction_costs_spot=TRANSACTION_COSTS_SPOT,
    transaction_costs_option=TRANSACTION_COSTS_OPTION,
    option_maturity=OPTION_MATURITY_3,
    forecast_horizon=H_3,
    spot_series=spot_curr_test_3,
    overnight_domestic_rate=overn_dom_r_test_3,
    overnight_foreign_rate=overn_for_r_test_3,
    domestic_rate=r_t_test_3,
    foreign_rate=r_b_test_3, 
    plots = False, 
    verbose=False, 
    sort_hyperparams_by=HYPERPARAM_SORT, 
    garch_1_2_indicator=GARCH_1_2_INDICATOR, 
    kbar=k_bar_MSM,
    b=b_MSM,
    gamma_kbar=gamma_kbar_MSM,
    convert_USD=CONVERT_USD_INDICATOR, 
    M = M, 
    long_thresholds=long_threshs,
    short_thresholds=short_threshs,
    sig_multipliers=sig_multipliers)

strategy_3.prepare_universal_series()
strategy_3.get_BMSM_data()
strategy_3.get_GARCH_data()
strategy_3.get_FIGARCH_data()

Estimated parameters: m0=1.210864e+00, sigma_bar=4.854110e-01
Final log-likelihood: -8.041643e+02
Estimated parameters: m0=1.199194e+00, sigma_bar=4.851642e-01
Final log-likelihood: -1.292050e+03
Estimated parameters: omega=0.0009705549253154421, alpha=0.0378, beta=0.9598
Estimated parameters: omega=0.0011646355172409558, alpha=0.0403, beta=0.9564
Estimated parameters: omega=0.04783632598877084, d=0.2740, beta=0.2590
Final log-likelihood = 598.9679
Estimated parameters: omega=0.04355291536306382, d=0.2798, beta=0.2660
Final log-likelihood = 972.5349


In [28]:
error_metrics_df_3, _, m_z_results_3, _ = strategy_3.in_sample_predictions()

In [29]:
error_metrics_df_3

,Norm MSE,DM Test Stat MSE,DM p-value (one-sided) MSE,Norm MAE,DM Test Stat MAE,DM p-value (one-sided) MAE
BMSM,0.906842,NaN,NaN,0.976826,NaN,NaN
BMSM OLS,0.944898,0.220441,0.587215,0.924406,-0.395695,0.346205
GARCH,0.953232,NaN,NaN,1.015310,NaN,NaN
GARCH OLS,0.833796,-0.358334,0.360083,0.861666,-0.877172,0.190297
FIGARCH,0.975328,NaN,NaN,1.066274,NaN,NaN
FIGARCH OLS,0.979277,0.012210,0.504870,0.883901,-0.880870,0.189295


In [30]:
m_z_results_3

{'BMSM': {'alpha_hat': -0.0011828304572032173,
  'beta_hat': 1.2428344642090836,
  'alpha_p': 0.43346347398037366,
  'beta_p': 0.3563563922604627},
 'GARCH': {'alpha_hat': 0.0023313251867395483,
  'beta_hat': 0.5646745078457295,
  'alpha_p': 0.00018526580847849283,
  'beta_p': 1.0380711442498255e-05},
 'FIGARCH': {'alpha_hat': 0.0010398556547353327,
  'beta_hat': 0.7614206253739209,
  'alpha_p': 0.40118351743752945,
  'beta_p': 0.2035711018445009}}

In [31]:
strategy_3.model_bmsm.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                      y   R-squared:                       0.297
Model:                            OLS   Adj. R-squared:                  0.294
Method:                 Least Squares   F-statistic:                     14.61
Date:                Sat, 30 Aug 2025   Prob (F-statistic):           1.29e-11
Time:                        20:56:35   Log-Likelihood:                -181.96
No. Observations:                1051   AIC:                             373.9
Df Residuals:                    1046   BIC:                             398.7
Df Model:                           4                                         
Covariance Type:                  HAC                                         
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.0672      0.023     -2.918      0.004      -0.112      -0.022
x1             0.1030      0.052      2.000      0.046       0.002       0.204
x2            -0.0060      0.040     -0.151      0.880      -0.084       0.072
x3            -0.0027      0.023     -0.116      0.908      -0.048       0.042
x4            -0.0973      0.050     -1.962      0.050      -0.194      -0.000
==============================================================================
Omnibus:                        2.783   Durbin-Watson:                   0.028
Prob(Omnibus):                  0.249   Jarque-Bera (JB):                2.532
Skew:                          -0.049   Prob(JB):                        0.282
Kurtosis:                       2.781   Cond. No.                         6.80
==============================================================================

Notes:
[1] Standard Errors are heteroscedasticity and autocorrelation robust (HAC) using 6 lags and without small sample correction
"""

In [32]:
strategy_3.model_garch.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                      y   R-squared:                       0.038
Model:                            OLS   Adj. R-squared:                  0.035
Method:                 Least Squares   F-statistic:                     2.187
Date:                Sat, 30 Aug 2025   Prob (F-statistic):             0.0879
Time:                        20:57:04   Log-Likelihood:                -157.78
No. Observations:                1051   AIC:                             323.6
Df Residuals:                    1047   BIC:                             343.4
Df Model:                           3                                         
Covariance Type:                  HAC                                         
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.1989      0.023     -8.834      0.000      -0.243      -0.155
x1            -0.0662      0.026     -2.535      0.011      -0.117      -0.015
x2             0.0062      0.021      0.292      0.771      -0.035       0.048
x3            -0.0793      0.039     -2.056      0.040      -0.155      -0.004
==============================================================================
Omnibus:                        5.074   Durbin-Watson:                   0.037
Prob(Omnibus):                  0.079   Jarque-Bera (JB):                4.018
Skew:                          -0.032   Prob(JB):                        0.134
Kurtosis:                       2.704   Cond. No.                         3.41
==============================================================================

Notes:
[1] Standard Errors are heteroscedasticity and autocorrelation robust (HAC) using 6 lags and without small sample correction
"""

In [33]:
strategy_3.model_figarch.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                      y   R-squared:                       0.301
Model:                            OLS   Adj. R-squared:                  0.299
Method:                 Least Squares   F-statistic:                     19.29
Date:                Sat, 30 Aug 2025   Prob (F-statistic):           3.55e-12
Time:                        20:57:06   Log-Likelihood:                -226.16
No. Observations:                1051   AIC:                             460.3
Df Residuals:                    1047   BIC:                             480.1
Df Model:                           3                                         
Covariance Type:                  HAC                                         
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.1630      0.024     -6.802      0.000      -0.210      -0.116
x1            -0.0181      0.027     -0.667      0.505      -0.071       0.035
x2             0.0369      0.024      1.542      0.123      -0.010       0.084
x3            -0.2288      0.036     -6.427      0.000      -0.299      -0.159
==============================================================================
Omnibus:                        7.969   Durbin-Watson:                   0.032
Prob(Omnibus):                  0.019   Jarque-Bera (JB):                7.941
Skew:                          -0.211   Prob(JB):                       0.0189
Kurtosis:                       3.053   Cond. No.                         2.74
==============================================================================

Notes:
[1] Standard Errors are heteroscedasticity and autocorrelation robust (HAC) using 6 lags and without small sample correction
"""

# H=364, T=1Y

In [34]:
OPTION_MATURITY_4 = 1
H_4 = 364

vol_smile_data_4 = pd.read_csv("eurusd_vol_smile_1y_extra.csv").set_index("CalculationDate")
implied_vol_data_4 = pd.DataFrame({
    'Exchange Date': vol_smile_data_4.index, 
    "Bid": vol_smile_data_4['ATM'], 
    "Ask": vol_smile_data_4['ATM'],
    "BidNet": vol_smile_data_4['ATM']})
Data_clean_4 = Clean_Implied_Vols_EUR_with_smile(data=implied_vol_data_4, 
                                start_date=start_date, 
                                end_date=end_date, 
                                align_df1=realised_variance_eurusd, 
                                align_df2=daily_log_returns_eurusd, 
                                smile_df=vol_smile_data_4)
implied_vol_data_4, realised_variance_4, daily_log_returns_4, vol_smile_data_4 = Data_clean_4.get_clean_data()

N_4 = len(daily_log_returns_4)
test_align_4 = daily_log_returns_4.iloc[N_4//2:-H_4]
spot_curr_test_4 = align_spots(spot_curr, test_align_4, start_date, end_date)

r_b_4 = pd.read_csv("estr_1y_compounded.csv").set_index("Date")
r_t_4 = pd.read_csv("/Users/alexvillamartin/Documents/MSc Diss/Code/SOFR_1y_compounded.csv").set_index("date")[['rate_pct']]

r_b_test_4, r_t_test_4, overn_dom_r_test_4, overn_for_r_test_4= sort_rates(r_b_4, r_t_4, overnight_domestic_rate, test_align_4, overnight_foreign_rate) 
r_b_test_4.ffill(inplace=True)

In [35]:
strategy_4 = Compare_Trading_Strategies(
    return_series=daily_log_returns_4, 
    realised_variance_series=realised_variance_4,
    atm_implied_vol_data=implied_vol_data_4,
    vol_smile_data=vol_smile_data_4,
    train_size=train_size,
    ticker=TICKER,
    initial_capital_domestic=IC_USD,
    notional_base=NOTIONAL_EUR,
    maximum_delta_difference=MAX_DELTA_DIFF,
    signal_lb=SIGNAL_LB,
    signal_ub=SIGNAL_UB,
    transaction_cost_indicator=TRANSACTION_COST_BOOL,
    transaction_costs_spot=TRANSACTION_COSTS_SPOT,
    transaction_costs_option=TRANSACTION_COSTS_OPTION,
    option_maturity=OPTION_MATURITY_4,
    forecast_horizon=H_4,
    spot_series=spot_curr_test_4,
    overnight_domestic_rate=overn_dom_r_test_4,
    overnight_foreign_rate=overn_for_r_test_4,
    domestic_rate=r_t_test_4,
    foreign_rate=r_b_test_4, 
    plots = False, 
    verbose=False, 
    sort_hyperparams_by=HYPERPARAM_SORT, 
    garch_1_2_indicator=GARCH_1_2_INDICATOR, 
    kbar=k_bar_MSM,
    b=b_MSM,
    gamma_kbar=gamma_kbar_MSM,
    convert_USD=CONVERT_USD_INDICATOR, 
    M = M, 
    long_thresholds=long_threshs,
    short_thresholds=short_threshs,
    sig_multipliers=sig_multipliers)

strategy_4.prepare_universal_series()
strategy_4.get_BMSM_data()
strategy_4.get_GARCH_data()
strategy_4.get_FIGARCH_data()

Estimated parameters: m0=1.210864e+00, sigma_bar=4.854110e-01
Final log-likelihood: -8.041643e+02
Estimated parameters: m0=1.202056e+00, sigma_bar=4.886992e-01
Final log-likelihood: -1.192038e+03
Estimated parameters: omega=0.0009705549253154421, alpha=0.0378, beta=0.9598
Estimated parameters: omega=0.005056556141756255, alpha=0.0377, beta=0.9448
Estimated parameters: omega=0.04783632598877084, d=0.2740, beta=0.2590
Final log-likelihood = 598.9679
Estimated parameters: omega=0.04162535252925603, d=0.2939, beta=0.2833
Final log-likelihood = 938.3661


In [36]:
error_metrics_df_4, _, m_z_results_4, _ = strategy_4.in_sample_predictions()

In [38]:
error_metrics_df_4

,Norm MSE,DM Test Stat MSE,DM p-value (one-sided) MSE,Norm MAE,DM Test Stat MAE,DM p-value (one-sided) MAE
BMSM,1.167485,NaN,NaN,1.059821,NaN,NaN
BMSM OLS,2.523511,0.873373,0.808649,1.372486,0.560751,0.712444
GARCH,2.190068,NaN,NaN,1.401109,NaN,NaN
GARCH OLS,2.510693,0.117247,0.546654,1.343447,-0.058535,0.476668
FIGARCH,1.328984,NaN,NaN,1.180861,NaN,NaN
FIGARCH OLS,2.581887,0.581560,0.719493,1.387881,0.257927,0.601738


In [37]:
m_z_results_4

{'BMSM': {'alpha_hat': 0.014713116652441188,
  'beta_hat': -1.3878994778411944,
  'alpha_p': 8.046142515808623e-13,
  'beta_p': 2.8502455174143286e-12},
 'GARCH': {'alpha_hat': 0.0068972089454749945,
  'beta_hat': -0.060902139674765376,
  'alpha_p': 5.806514336589554e-28,
  'beta_p': 1.7382852257583418e-48},
 'FIGARCH': {'alpha_hat': 0.01330018053885773,
  'beta_hat': -0.9909168813661128,
  'alpha_p': 2.2770829697857713e-18,
  'beta_p': 1.400362226488767e-20}}

In [39]:
strategy_4.model_bmsm.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                      y   R-squared:                       0.688
Model:                            OLS   Adj. R-squared:                  0.686
Method:                 Least Squares   F-statistic:                     102.0
Date:                Sat, 30 Aug 2025   Prob (F-statistic):           4.07e-71
Time:                        21:03:52   Log-Likelihood:                 408.65
No. Observations:                 869   AIC:                            -807.3
Df Residuals:                     864   BIC:                            -783.5
Df Model:                           4                                         
Covariance Type:                  HAC                                         
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.0604      0.013     -4.650      0.000      -0.086      -0.035
x1             0.0074      0.021      0.357      0.721      -0.033       0.048
x2            -0.0135      0.018     -0.749      0.454      -0.049       0.022
x3             0.0254      0.019      1.337      0.181      -0.012       0.063
x4            -0.2456      0.027     -9.037      0.000      -0.299      -0.192
==============================================================================
Omnibus:                       21.292   Durbin-Watson:                   0.099
Prob(Omnibus):                  0.000   Jarque-Bera (JB):               21.625
Skew:                          -0.363   Prob(JB):                     2.02e-05
Kurtosis:                       2.735   Cond. No.                         4.94
==============================================================================

Notes:
[1] Standard Errors are heteroscedasticity and autocorrelation robust (HAC) using 6 lags and without small sample correction
"""

In [40]:
strategy_4.model_garch.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                      y   R-squared:                       0.451
Model:                            OLS   Adj. R-squared:                  0.449
Method:                 Least Squares   F-statistic:                     34.02
Date:                Sat, 30 Aug 2025   Prob (F-statistic):           8.72e-21
Time:                        21:03:54   Log-Likelihood:                 374.21
No. Observations:                 869   AIC:                            -740.4
Df Residuals:                     865   BIC:                            -721.4
Df Model:                           3                                         
Covariance Type:                  HAC                                         
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.3225      0.013    -24.162      0.000      -0.349      -0.296
x1            -0.1302      0.015     -8.433      0.000      -0.160      -0.100
x2             0.0252      0.020      1.273      0.203      -0.014       0.064
x3            -0.2190      0.027     -8.133      0.000      -0.272      -0.166
==============================================================================
Omnibus:                       29.525   Durbin-Watson:                   0.171
Prob(Omnibus):                  0.000   Jarque-Bera (JB):               30.387
Skew:                          -0.432   Prob(JB):                     2.52e-07
Kurtosis:                       2.698   Cond. No.                         3.54
==============================================================================

Notes:
[1] Standard Errors are heteroscedasticity and autocorrelation robust (HAC) using 6 lags and without small sample correction
"""

In [41]:
strategy_4.model_figarch.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                      y   R-squared:                       0.648
Model:                            OLS   Adj. R-squared:                  0.646
Method:                 Least Squares   F-statistic:                     86.05
Date:                Sat, 30 Aug 2025   Prob (F-statistic):           9.92e-49
Time:                        21:03:57   Log-Likelihood:                 329.41
No. Observations:                 869   AIC:                            -650.8
Df Residuals:                     865   BIC:                            -631.8
Df Model:                           3                                         
Covariance Type:                  HAC                                         
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.1876      0.014    -13.309      0.000      -0.215      -0.160
x1            -0.0253      0.013     -1.947      0.052      -0.051       0.000
x2             0.0229      0.021      1.105      0.269      -0.018       0.064
x3            -0.2428      0.021    -11.590      0.000      -0.284      -0.202
==============================================================================
Omnibus:                        1.873   Durbin-Watson:                   0.122
Prob(Omnibus):                  0.392   Jarque-Bera (JB):                1.746
Skew:                           0.069   Prob(JB):                        0.418
Kurtosis:                       3.171   Cond. No.                         2.51
==============================================================================

Notes:
[1] Standard Errors are heteroscedasticity and autocorrelation robust (HAC) using 6 lags and without small sample correction
"""